# 🧠 Medical Chatbot using RAG Architecture

This notebook builds a **Medical Question Answering Chatbot** using:

- LangChain
- Pinecone Vector Database
- HuggingFace Embeddings
- OpenAI LLM

---

## 📌 What is RAG?

RAG (Retrieval Augmented Generation) means:

1. Search relevant information from documents
2. Send that information to LLM
3. Generate accurate answers

Instead of guessing, the model answers using real data.


In [2]:
# ==========================================================
# STEP 1: Basic Environment Check
# ==========================================================

# Print confirmation message
print("Notebook is running successfully")

# Show current working directory
%pwd


Notebook is running successfully


'f:\\AI Projects\\End-to-End-GenAI-RAG-Based-Application-Medical-Chatbot\\notebook_experiment'

In [3]:
import os
os.chdir("../")
%pwd

'f:\\AI Projects\\End-to-End-GenAI-RAG-Based-Application-Medical-Chatbot'

## 📂 Step 2 — Import Required Libraries

We import modules required for:

- Loading PDFs
- Processing documents
- Creating embeddings
- Connecting vector database


In [4]:
# ==========================================================
# STEP 2: Import Libraries
# ==========================================================

import os

# Document loaders
from langchain.document_loaders import PyPDFLoader, DirectoryLoader


## 📄 Step 3 — Load PDF Documents

We load all medical PDFs stored inside the `data/` folder.

Each PDF becomes a **Document object**.


In [5]:
# ==========================================================
# STEP 3: Load PDFs
# ==========================================================

def load_pdf_files(data_path):
    """
    Loads all PDF files from a directory.

    Parameters:
        data_path (str): folder containing PDFs

    Returns:
        list: extracted documents
    """
    
    loader = DirectoryLoader(
        data_path,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    
    documents = loader.load()
    return documents


# Load documents
extracted_data = load_pdf_files("data")

print("Total documents loaded:", len(extracted_data))


Total documents loaded: 637


## ✂️ Step 4 — Split Documents into Chunks

Large documents are divided into smaller parts.

### Why?

LLMs cannot process very long text efficiently.

Chunking improves:
- search accuracy
- retrieval speed
- answer quality


In [6]:
# ==========================================================
# STEP 4: Text Chunking
# ==========================================================

from langchain.text_splitter import RecursiveCharacterTextSplitter

def split_documents(documents):
    """
    Splits documents into smaller chunks.
    """
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,     # characters per chunk
        chunk_overlap=100   # overlapping characters
    )

    chunks = text_splitter.split_documents(documents)
    return chunks


texts_chunk = split_documents(extracted_data)

print("Number of chunks created:", len(texts_chunk))


Number of chunks created: 6600


## 🔢 Step 5 — Create Embeddings

Embeddings convert text into numbers.

Example:

"What is Acne?"
↓
[0.12, -0.44, 0.91, ...]  (384 numbers)

These numbers help computers understand meaning.


In [7]:
# ==========================================================
# STEP 5: Load Embedding Model
# ==========================================================

from langchain.embeddings import HuggingFaceEmbeddings

def load_embeddings():
    """
    Loads sentence transformer embedding model.
    """
    
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    
    return embeddings


embedding = load_embeddings()

# Test embedding
vector = embedding.embed_query("Hello world")

print("Embedding dimension:", len(vector))


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_2540\3135453902.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Embedding dimension: 384


## 🔐 Step 6 — Load API Keys

We use `.env` file to securely store keys.


In [10]:
# ==========================================================
# STEP 6: Load Environment Variables
# ==========================================================

from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


## 🗄 Step 7 — Connect to Pinecone Vector Database

Vector DB stores embeddings and enables semantic search.


In [11]:
# ==========================================================
# STEP 7: Pinecone Connection
# ==========================================================

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medical-chatbot"

# Create index if it doesn't exist
if index_name not in pc.list_indexes().names():
    
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )


## 📥 Step 8 — Store Embeddings in Vector Database


In [ ]:
# ==========================================================
# STEP 8: Upload Documents to Pinecone
# ==========================================================

from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

print("Documents stored successfully!")


## 🔎 Step 9 — Create Retriever

Retriever finds most relevant chunks based on question meaning.


In [ ]:
# ==========================================================
# STEP 9: Retriever
# ==========================================================

retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # return top 3 results
)


## 🤖 Step 10 — Load LLM


In [ ]:
# ==========================================================
# STEP 10: Load Chat Model
# ==========================================================

from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4o")


## 🔗 Step 11 — Build RAG Chain


In [ ]:
# ==========================================================
# STEP 11: Create RAG Pipeline
# ==========================================================

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful medical assistant. Use context to answer."),
    ("human", "{input}")
])

qa_chain = create_stuff_documents_chain(chat_model, prompt)

rag_chain = create_retrieval_chain(retriever, qa_chain)


## 💬 Step 12 — Ask Questions


In [ ]:
# ==========================================================
# STEP 12: Query the Chatbot
# ==========================================================

response = rag_chain.invoke({
    "input": "What is Acne?"
})

print(response["answer"])


# ✅ Final Pipeline Summary

PDF → Chunk → Embedding → Pinecone → Retriever → LLM → Answer

This is a complete **RAG-based AI application**.
